# Сравнение сочетаний функций при изменении назначения территории

Ноутбук считает метод от исходных слоёв до таблицы сравнения вариантов. Вручную ничего считать не нужно.

**Порядок работы**
1. Положите свои слои в папку `data` рядом с ноутбуком.
2. Заполните анкету в ячейке **«Шаг 0. Ваши данные»**: пути к файлам и имена полей.
3. Выполните все ячейки (**Run All**). Ячейка **«Шаг 0б. Проверка данных»** покажет, что найдено в слоях. Если есть ошибки (✕), расчёт остановится и подскажет, что исправить.
4. Результаты появятся в папке `results`.

**Какие слои нужны**

| Слой | Геометрия | Какие поля нужны |
|---|---|---|
| Кварталы | полигоны | номер квартала (можно без него — пронумеруются по порядку) |
| Здания | полигоны | жилое или нет; этажность; по желанию — площадь застройки, жилая площадь, жители |
| Сервисы | точки или полигоны | тип объекта; у школ и детских садов — число мест |
| Зоны (по желанию) | полигоны | код зоны ПЗЗ или функциональной зоны |

Тип сервиса относится к группе по файлу **`таблица_групп_сервисов.csv`** (открывается в Excel). Правило таблицы — код ВРИ объекта:
- **К** — 3.3, 4.2, 4.3, 4.4, 4.6;
- **ОД** — 3.2.3, 3.5.2, 3.6.1, 3.8, 4.1, 4.5, 4.7;
- **П** — 6.0–6.12;
- школы и детские сады учитываются по числу мест, метро и остановки — по расстоянию до территории;
- остальные типы — группа «нет», в расчёте не участвуют.

Если в ваших данных есть тип, которого нет в таблице, проверка покажет его. Тогда добавьте строку: тип, код ВРИ, группа.

**Что нужно установить.** Python 3.10+ и библиотеки numpy, pandas, matplotlib, pyyaml. Все они есть в Anaconda и Google Colab.

**Расчёт выполняет пакет `mixuse`** — папка `mixuse/` должна лежать рядом с ноутбуком (в Google Colab загрузите её вместе с ноутбуком). Ноутбук заполняет анкету, вызывает функции пакета по шагам и показывает результаты. Те же расчёты выполняет командная строка: `python -m mixuse compare configs/<код>.yaml`.

## Шаг 0. Ваши данные

Заполните анкету: впишите пути к файлам и имена полей **так, как они называются в вашем слое** (их видно в QGIS: «Свойства слоя» → «Поля»). Если поля нет, оставьте `None`.

- Пути пишутся от папки, где лежит ноутбук.
- Форматы файлов: GeoJSON (`.geojson`) или GeoPackage (`.gpkg`).
- Все слои должны быть в одной системе координат: EPSG:4326 (градусы) или метрической (UTM, МСК). Web Mercator (EPSG:3857) не подходит.

**Сервисы** можно дать двумя способами:
- **папкой**, где один файл — один тип, а имя файла и есть тип: `school.geojson`, `cafe.geojson`… Так устроены данные blocksnet;
- **одним файлом** с полем типа. Для выгрузки из OpenStreetMap укажите список полей тегов, например `['amenity', 'shop', 'office', 'tourism']`. Тип тогда записывается как тег: `amenity=school`. Такие теги указаны в столбце «тег_OSM» таблицы групп.

In [ ]:
# ===== ШАГ 0. ВАШИ ДАННЫЕ — заполняется только эта ячейка =====
ДАННЫЕ = dict(
    название='Мой город',
    код='moy_gorod',                  # короткое имя для файлов результатов (латиницей, без пробелов)
    режим='слои',                     # 'слои' — кварталы, здания и сервисы отдельными слоями
                                      # 'таблица' — к кварталам уже присоединены здания и сервисы (см. пример Гатчины)
    кварталы=dict(
        файл='data/moy_gorod/кварталы.geojson',
        поле_номера='block_id',       # поле с номером квартала; None — пронумеровать по порядку
    ),
    здания=dict(
        файл='data/moy_gorod/здания.geojson',
        поле_жилое='is_living',       # жилое ли здание: 1/0, True/False, да/нет
        поле_этажность='storeys',     # число этажей; если пусто — 1 этаж
        поле_площадь_застройки=None,  # None — по контуру здания
        поле_жилая_площадь=None,      # None — поэтажная площадь × k_liv
        поле_жители=None,             # None — жилая площадь ÷ q
    ),
    сервисы=dict(
        файл='data/moy_gorod/сервисы',   # папка (один файл = один тип), один файл или список: ['…/сервисы', '…/промышленность']
        поле_типа=None,               # для одного файла: 'type' или список полей OSM ['amenity', 'shop', 'office', 'tourism']
        поле_мест='capacity',         # число мест у школ и детских садов
    ),
    зоны=dict(                        # необязательно: если зон нет, оставьте файл=None
        файл=None,
        поле_кода=None,
        правила=[('ТП', 'П'), ('ТД', 'ОД')],   # нежилое здание без сервисов: код зоны начинается с «ТП» → П, «ТД» → ОД, иначе прочее
    ),
    территория=[],                    # номера кварталов, где меняется назначение, например [322, 323, 377]
)
ТАБЛИЦА_ГРУПП = 'таблица_групп_сервисов.csv'

### Примеры заполнения

Так анкета заполнена для трёх наборов данных, на которых проверялся метод. Чтобы посчитать пример, поставьте `ПРИМЕР = 'spb'`, `ПРИМЕР = 'vysotsk'` или `ПРИМЕР = 'gatchina'`. Для своих данных — `ПРИМЕР = None`.

- **spb** — Васильевский остров: отдельные слои кварталов, зданий, сервисов (папка файлов) и функциональных зон.
- **vysotsk** — Высоцк: отдельные слои кварталов, зданий, сервисов (две папки: сервисы и промышленность) и зон из генплана.
- **gatchina** — Гатчина: готовая таблица кварталов. Сервисы и здания уже присоединены к кварталам авторами данных, поэтому нежилая площадь относится к группе по зоне квартала, а не по сервисам внутри здания.

In [ ]:
ПРИМЕР = None    # None — ваши данные из шага 0; 'spb', 'vysotsk' или 'gatchina' — пример

ПРИМЕРЫ = {
    'spb': dict(
        название='Санкт-Петербург, Васильевский остров', код='spb', режим='слои',
        кварталы=dict(файл='examples/spb_vasilievsky/blocks.geojson', поле_номера='block_id'),
        здания=dict(файл='examples/spb_vasilievsky/buildings.geojson', поле_жилое='is_living', поле_этажность='storeys_count',
                    поле_площадь_застройки='building_area', поле_жилая_площадь='living_area', поле_жители='population_balanced'),
        сервисы=dict(файл='examples/spb_vasilievsky/services', поле_типа=None, поле_мест='capacity'),
        зоны=dict(файл='examples/spb_vasilievsky/functional_zones.geojson', поле_кода='functional_zone', правила=[('ТП', 'П'), ('ТД', 'ОД')]),
        территория=[8346, 9366, 2059],          # промзона у наб. Макарова
    ),
    'gatchina': dict(
        название='Гатчина', код='gatchina', режим='таблица',
        кварталы=dict(файл='examples/gatchina/blocks_with_services.gpkg', поле_номера='block_id'),
        готовая_таблица=dict(поле_жители='population', поле_жилая_площадь='living_area', поле_нежилая_площадь='non_living_area',
                             поле_зоны='land_use', префикс_числа='count_', префикс_мест='capacity_',
                             правила=[('LandUse.INDUSTRIAL', 'П'), ('LandUse.BUSINESS', 'ОД')]),
        территория=[322, 323, 377],             # промзона
    ),
    'vysotsk': dict(
        название='Высоцк', код='vysotsk', режим='слои',
        кварталы=dict(файл='examples/vysotsk/кварталы.geojson', поле_номера=None),   # кварталы пересохранены в EPSG:4326, номер — по порядку
        здания=dict(файл='examples/vysotsk/здания.geojson', поле_жилое='is_living', поле_этажность='number_of_floors',
                    поле_площадь_застройки='footprint_area', поле_жилая_площадь='living_area', поле_жители='population'),
        сервисы=dict(файл=['examples/vysotsk/сервисы', 'examples/vysotsk/промышленность'], поле_типа=None, поле_мест='capacity'),
        зоны=dict(файл='examples/vysotsk/зоны.geojson', поле_кода='Зона', правила=[('Производственная', 'П'), ('Общественно-деловая', 'ОД')]),
        территория=[362, 387, 392],             # производственные кварталы у жилой застройки (пример)
    ),
}

## Параметры метода и варианты

- `P` — параметры. Значения — ориентиры; их источники указываются в НИР.
- `SCN` — варианты сочетаний: доли поэтажной площади в процентах.

In [ ]:
P = dict(
    kit=1.4,            # КИТ: поэтажная площадь / площадь территории
    R=800,              # радиус окружения, м
    r=500,              # радиус пешей доступности функции, м
    q=30,               # м² жилой площади на человека
    k_liv=0.7,          # доля жилой площади в поэтажной
    a_К=35, a_ОД=20, a_П=70, a_X=50,   # м² поэтажной площади на одно рабочее место
    n_school=110, n_kinder=60,          # мест в школах и детских садах на 1000 жителей
    S_min=200,          # нежилые здания меньше этой площади застройки не учитываются, м²
    storeys_default=1,  # этажность, если она не указана
    builtin=True,       # учитывать встроенные помещения: сервисы К и ОД внутри контура жилого дома
    builtin_s0=100,     # площадь одного встроенного помещения, м² (допущение; проверка 50 и 300)
    builtin_share=0.5,  # встроенные помещения занимают не больше этой доли первого этажа (уличная сторона)
    builtin_min_storeys=2,   # правило применяется к жилым домам не ниже стольких этажей
)
SCN = {   # доли поэтажной площади, %: Ж — жильё, К — торговля и услуги, ОД — общественно-деловая, П — производство
    'С0 Ж+К':  {'Ж': 85, 'К': 15, 'ОД': 0,  'П': 0},
    'С1 Ж+ОД': {'Ж': 60, 'К': 10, 'ОД': 30, 'П': 0},
    'С2 ОД+П': {'Ж': 0,  'К': 10, 'ОД': 40, 'П': 50},
    'С3 Ж+П':  {'Ж': 50, 'К': 10, 'ОД': 0,  'П': 40},
}
OUT = 'results'

## Функции метода — пакет `mixuse`

Чтение слоёв, перевод градусов в метры, проверка «точка внутри полигона», расстояния и формулы шагов 1–9 находятся в пакете `mixuse` (папка рядом с ноутбуком). Эту ячейку менять не нужно.

| Модуль | Что делает |
|---|---|
| `mixuse.geometry` | чтение GeoJSON и GeoPackage, перевод в метры, площадь, точка в полигоне, расстояния |
| `mixuse.groups` | таблица групп сервисов |
| `mixuse.blocks` | шаги 1–4: таблица кварталов (`load_layers`, `build_table`) |
| `mixuse.compare` | шаги 5–6: окружение и сравнение вариантов (`compare`) |
| `mixuse.checks` | проверка данных; шаги 7 и 7б (`sensitivity`, `builtin_sensitivity`) |
| `mixuse.export` | таблицы CSV, слой для QGIS, карта, файл для страницы |
| `mixuse.notebook` | перевод анкеты шага 0 в настройки пакета, печать проверки |

In [ ]:
# ===== ФУНКЦИИ МЕТОДА — из пакета mixuse (менять не нужно) =====
import os, time
import pandas as pd
import matplotlib.pyplot as plt
from mixuse.notebook import config_from_form, print_check
from mixuse.blocks import load_layers, build_table
from mixuse.compare import compare
from mixuse.checks import sensitivity, builtin_sensitivity
from mixuse import export

## Шаг 0б. Проверка данных

Ячейка открывает слои и печатает, что в них нашлось:
- ✓ — всё в порядке;
- ! — предупреждение: расчёт пойдёт, но результат стоит проверить;
- ✕ — ошибка: расчёт остановится.

Проверяется:
- есть ли файлы и поля из анкеты;
- уникальны ли номера кварталов и есть ли кварталы территории;
- сколько жилых зданий и у скольких нет этажности;
- какие типы сервисов попали в группы К, ОД, П;
- какие типы не найдены в таблице групп;
- есть ли число мест у школ и детских садов;
- одна ли система координат у всех слоёв.

In [ ]:
Д = ПРИМЕРЫ[ПРИМЕР] if ПРИМЕР else ДАННЫЕ
cfg = config_from_form(Д, P, SCN, ТАБЛИЦА_ГРУПП)          # анкета → настройки пакета
P, SCN = cfg['params'], cfg['scenarios']                  # параметры и варианты, проверенные пакетом
print(f'Набор данных: {cfg["title"]} (режим «{Д.get("режим", "слои")}»)\n')
print_check(cfg)

## Шаги 1–4. Таблица кварталов

1. **Слои переводятся в метры.**
2. **Каждый сервис получает группу** по таблице групп.
3. **Каждое здание и сервис сводится к точке** внутри него и относится к кварталу, где лежит эта точка.
4. **По каждому кварталу считаются:**
    - **жители** — из атрибута; если его нет, жилая площадь ÷ q;
    - **нежилая площадь по группам.** Поэтажная площадь = площадь застройки × этажность. Группа здания определяется по сервисам внутри его контура с приоритетом П → ОД → К; если сервисов нет — по зоне квартала;
    - **встроенные помещения.** Если внутри контура жилого дома (не ниже 2 этажей) есть сервисы групп К или ОД, часть его первого этажа считается нежилой: площадь = число сервисов × `builtin_s0` (100 м²), но не больше доли `builtin_share` (0,5) площади застройки — уличной стороны этажа. Она делится между К и ОД пропорционально числу сервисов и добавляется к нежилой площади этих групп (столбцы `встр_К`, `встр_ОД`). Жители дома не меняются. Отключается параметром `builtin=False`;
    - **рабочие места** = нежилая площадь группы ÷ площадь на одно место (a_К, a_ОД, a_П, a_X);
    - **есть ли функция К, ОД, П** — есть сервис этой группы или нежилая площадь этой группы;
    - **места в школах и детских садах** — сумма мест.

In [ ]:
os.makedirs(OUT, exist_ok=True)
КОД = cfg['name']; t0 = time.time()
L = load_layers(cfg, cfg['_groups'])          # слои, приведённые к системе координат кварталов
city = build_table(L, cfg, P, cfg['_groups'])
print(f'Подготовка заняла {time.time() - t0:.0f} с')
for note in city.get('crs_notes', []):
    print(f'  пересчёт координат: {note}')
for k, v in city['report'].items():
    print(f'  {k}: {v}')
export.blocks_csv(city, f'{OUT}/{КОД}_таблица_кварталов.csv')
tab = city['table']
tab[['block_id', 'zone', 'area', 'pop', 'jobs', 'есть_К', 'есть_ОД', 'есть_П', 'cap_school', 'cap_kinder']].round(0).head(8)

## Шаги 5–6. Окружение и сравнение вариантов

5. **Окружение** — кварталы, центр которых не дальше R от границы территории.
6. **Для каждого варианта:**
    - **что появится на территории:**
        - F = площадь территории × КИТ;
        - новые жители N = F × доля жилья × k_liv ÷ q;
        - рабочие места J = F × Σ (доля группы ÷ площадь на место);
    - **изменение рабочих мест** — J варианта минус рабочие места территории в нынешнем виде;
- **работа на жителя** K считается в двух масштабах: окружение — (рабочие места окружения + J) ÷ (жители окружения + N); все данные — то же по всем загруженным кварталам. Столбец «Как есть» — значение до изменения, столбцы вариантов — после. Среднее по данным K_ref приводится справочно; предпочтительный вариант по этому показателю метод не называет;
    - **доступ** — у скольких жителей окружения функция появилась или пропала в радиусе r;
    - **школы и детские сады:**
        - ученики (дети) с территории = N × норма ÷ 1000;
        - потребность = (жители окружения + N) × норма ÷ 1000;
        - обеспеченность окружения = места в окружении ÷ потребность × 100 %;
        - не хватает мест = потребность − места, если она больше мест.

В вариантах всё существующее на территории считается снесённым, в том числе школы и детские сады. Новые школы на территории в расчёт не входят: строка «не хватает мест» показывает, сколько мест нужно предусмотреть в проекте.

In [ ]:
T = cfg['territory']
table, info, det = compare(city, T, P, SCN)
for k, v in info.items():
    print(f'{k}: {v}')
export.table_csv(table, f'{OUT}/{КОД}_сравнение_вариантов.csv')
table

## Шаг 7. Проверка устойчивости

Расчёт повторяется при других радиусах окружения и КИТ. Для каждого сочетания показано, на сколько вариант меняет K окружения относительно «как есть». Если знак изменения сохраняется, а величина меняется мало, вывод не зависит от выбранных значений. Для школ показано, сколько мест не хватает в окружении.

In [ ]:
stab = sensitivity(city, T, P, SCN)          # R = 500, 800, 1000 м × КИТ = 1,4 и 2,0
export.table_csv(stab, f'{OUT}/{КОД}_устойчивость.csv', index=False)
stab

## Шаг 7б. Проверка допущения о встроенных помещениях

Площадь одного встроенного помещения `builtin_s0` — допущение. Таблица кварталов и сравнение вариантов пересчитываются без встроенных помещений и при s₀ = 50, 100, 300 м². Если K окружения меняется мало, вывод от этого допущения не зависит. Только для режима «слои».


In [ ]:
bi = builtin_sensitivity(L, cfg, P, T, SCN)   # без встроенных и s0 = 50, 100, 300 м²
if bi is not None:
    export.table_csv(bi, f'{OUT}/{КОД}_встроенные.csv', index=False)
    display(bi) if 'display' in globals() else print(bi.to_string(index=False))
else:
    print('Режим «таблица»: слоя зданий нет, встроенные помещения не учитываются.')

## Шаг 8. Карта и слой для QGIS

На карте: территория — фиолетовая, окружение обведено. Для выбранного варианта жилые кварталы окрашены по изменению доступа: зелёный — функции появились, оранжевый — пропали. Тот же результат сохраняется в слой `results/<город>_кварталы.geojson`. Его можно открыть в QGIS и оформить для рисунков НИР.

In [ ]:
SHOW = 'С1 Ж+ОД'   # какой вариант показать на карте

fig = export.map_figure(city, det, SHOW, P['R'], cfg['title'])
fig.savefig(f'{OUT}/{КОД}_карта_{SHOW.split()[0]}.png', dpi=150, bbox_inches='tight')
plt.show()

export.blocks_geojson(city, f'{OUT}/{КОД}_кварталы.geojson', det, SCN)
print('Слой для QGIS сохранён:', f'{OUT}/{КОД}_кварталы.geojson')

## Шаг 9. Файл данных для веб-страницы

Ноутбук — «движок»: он готовит таблицу кварталов из исходных слоёв. Веб-страница — «витрина»: она загружает этот файл и при каждом выборе территории сама пересчитывает шаги 5–6 по тем же формулам. Файл сохраняется в формате, который читает страница «Конструктор смешанности»: кварталы с контурами и показателями, точки метро и остановок, параметры, варианты и территория из шага 0.

In [ ]:
export.web_json(city, cfg, f'{OUT}/{КОД}_для_страницы.json')
print('Файл для страницы:', f'{OUT}/{КОД}_для_страницы.json', '—', len(city['table']), 'кварталов')

## Как повторить вручную в QGIS

| Шаг | В ноутбуке | В QGIS |
|---|---|---|
| 1. Перевод в метры | `mixuse.geometry.LocalProjection` | «Перепроецировать слой» → WGS 84 / UTM zone 36N (EPSG:32636) |
| 2. Группы сервисов | `таблица_групп_сервисов.csv` | присоединить таблицу групп к слою сервисов по полю типа («Свойства слоя» → «Связи»), затем поле `группа` |
| 3. Привязка к кварталам | `mixuse.geometry.inner_point`, `assign_points` | «Точка на поверхности» → «Объединить атрибуты по расположению (сводка)» с суммой или числом |
| 4. Поэтажная площадь, жители | `mixuse.blocks.table_from_layers` | «Калькулятор полей»: `building_area * storeys`, `living_area / 30` |
| 4. Группа нежилого здания | сервисы внутри контура: П → ОД → К | «Подсчёт точек в полигоне» по зданиям для каждой группы, затем CASE |
| 4. Встроенные помещения | `min(n · s0, 0,5 · S)` для жилого дома с сервисами К, ОД | «Подсчёт точек в полигоне» по жилым зданиям для К и ОД → «Калькулятор полей»: `min((n_К + n_ОД) * 100, 0.5 * building_area) * n_К / (n_К + n_ОД)` |
| 5. Окружение | `mixuse.compare.compare`: расстояние до границы ≤ R | выделить кварталы территории → «Буфер» R → «Центроиды» кварталов → «Выбрать по расположению» |
| 6. Доступ | расстояние ≤ r | «Буфер» r вокруг центроида жилого квартала → «Выбрать по расположению» кварталы с функцией |
| 6. Показатели | формулы в `compare` | «Статистика по полю» для сумм, формулы — в калькуляторе полей или Excel |